# Exploring Schemas: Browse Your Data Warehouse

## Overview

Before creating mappings, explore what data is available in your data warehouse. This notebook covers schema browsing, search, and metadata inspection.

**What you'll learn:**
- List catalogs, schemas, tables, and columns
- Search for tables and columns by pattern
- Access cache statistics (admin only)
- Handle schema metadata errors gracefully

**Prerequisites:**
- Completed: [03_query_cypher_basics.ipynb](./03_query_cypher_basics.ipynb)
- Data warehouse connection configured

**Time to complete:** 10 minutes

---

**Test Coverage:**
- Catalog/schema/table/column listing
- Table and column search
- Admin operations (stats, refresh)
- Cache performance validation

## Setup

Initialize SDK clients for regular user and admin user.

In [ ]:
# Parameters cell - papermill will inject values here
INSTANCE_ID = None  # Injected by papermill from fixtures

In [ ]:
import os
import time

from graph_olap import notebook
from graph_olap.testing import TestPersona
from graph_olap.exceptions import ForbiddenError, NotFoundError

print(f"GRAPH_OLAP_API_URL: {os.environ.get('GRAPH_OLAP_API_URL', 'not set')}")

# Wake up Starburst Galaxy cluster (auto-suspends after 5 min idle)
notebook.wake_starburst()

In [ ]:
# Create test context as analyst Alice for regular user tests
ctx = notebook.test("SchemaTest", persona=TestPersona.ANALYST_ALICE)
client = ctx.client

# Admin client for admin-only operations
admin_client = ctx.with_persona(TestPersona.ADMIN_CAROL)

print(f"Connected to {client._config.api_url}")
print(f"  - Regular user: analyst Alice")
print(f"  - Admin user: admin Carol")

## 1. Basic Listing Operations

Test catalog, schema, table, and column listing with error handling.

In [ ]:
# Test 1.1: List all available catalogs
catalogs = client.schema.list_catalogs()

assert isinstance(catalogs, list), "Should return list"
print(f"Found {len(catalogs)} catalogs")

if catalogs:
    print("\nFirst 5 catalogs:")
    for cat in catalogs[:5]:
        print(f"  - {cat.catalog_name} ({cat.schema_count} schemas)")
else:
    print("  (Cache is empty - Starburst not configured)")

print("\nTest 1.1 PASSED: List catalogs")

In [ ]:
# Test 1.2: List schemas for non-existent catalog returns NotFoundError
try:
    schemas = client.schema.list_schemas("nonexistent_catalog")
    assert False, "Should have raised NotFoundError"
except NotFoundError as e:
    print(f"Correctly raised NotFoundError: {e}")

print("\nTest 1.2 PASSED: Error handling for missing catalog")

In [ ]:
# Test 1.3: List tables for non-existent schema returns NotFoundError
try:
    tables = client.schema.list_tables("catalog", "nonexistent_schema")
    assert False, "Should have raised NotFoundError"
except NotFoundError as e:
    print(f"Correctly raised NotFoundError: {e}")

print("\nTest 1.3 PASSED: Error handling for missing schema")

In [ ]:
# Test 1.4: List columns for non-existent table returns NotFoundError
try:
    columns = client.schema.list_columns("catalog", "schema", "nonexistent_table")
    assert False, "Should have raised NotFoundError"
except NotFoundError as e:
    print(f"Correctly raised NotFoundError: {e}")

print("\nTest 1.4 PASSED: Error handling for missing table")

## 2. Search Operations

Test table and column search with patterns and limits.

In [ ]:
# Test 2.1: Search tables by pattern with limit
results = client.schema.search_tables("test", limit=10)

assert isinstance(results, list), "Should return list"
assert len(results) <= 10, "Should respect limit"

print(f"Search found {len(results)} tables matching 'test' (limit=10)")
if results:
    print("\nFirst 3 results:")
    for r in results[:3]:
        print(f"  - {r.catalog_name}.{r.schema_name}.{r.table_name} ({r.table_type})")

print("\nTest 2.1 PASSED: Search tables with limit")

In [ ]:
# Test 2.2: Search columns by pattern with limit
results = client.schema.search_columns("id", limit=10)

assert isinstance(results, list), "Should return list"
assert len(results) <= 10, "Should respect limit"

print(f"Search found {len(results)} columns matching 'id' (limit=10)")
if results:
    print("\nFirst 3 results:")
    for r in results[:3]:
        print(f"  - {r.catalog_name}.{r.schema_name}.{r.table_name}.{r.column_name} ({r.data_type})")

print("\nTest 2.2 PASSED: Search columns with limit")

## 3. Admin Operations

Test admin-only operations with permission checks.

In [ ]:
# Test 3.1: Regular users cannot access admin endpoints
# Test get_stats forbidden for regular user
try:
    stats = client.schema.get_stats()
    assert False, "Should have raised ForbiddenError"
except ForbiddenError as e:
    print(f"Correctly raised ForbiddenError for stats: {e}")

# Test admin_refresh forbidden for regular user
try:
    result = client.schema.admin_refresh()
    assert False, "Should have raised ForbiddenError"
except ForbiddenError as e:
    print(f"Correctly raised ForbiddenError for refresh: {e}")

print("\nTest 3.1 PASSED: Permission checks")

In [ ]:
# Test 3.2: Get cache statistics as admin
stats = admin_client.schema.get_stats()

assert stats.total_catalogs >= 0, "Should have non-negative catalog count"
assert stats.total_schemas >= 0, "Should have non-negative schema count"
assert stats.total_tables >= 0, "Should have non-negative table count"
assert stats.total_columns >= 0, "Should have non-negative column count"
assert stats.index_size_bytes >= 0, "Should have non-negative index size"

print("Cache statistics (as admin):")
print(f"  - Catalogs: {stats.total_catalogs}")
print(f"  - Schemas: {stats.total_schemas}")
print(f"  - Tables: {stats.total_tables}")
print(f"  - Columns: {stats.total_columns}")
print(f"  - Index size: {stats.index_size_bytes} bytes")
if stats.last_refresh:
    print(f"  - Last refresh: {stats.last_refresh}")

print("\nTest 3.2 PASSED: Get stats as admin")

In [ ]:
# Test 3.3: Trigger manual cache refresh as admin
result = admin_client.schema.admin_refresh()

assert isinstance(result, dict), "Should return dict"
assert result["status"] == "refresh triggered", "Should confirm trigger"

print(f"Refresh triggered: {result}")
print("\nTest 3.3 PASSED: Admin refresh trigger")

## 4. Complete Workflows

Test complete end-to-end user workflows for schema browsing.

In [ ]:
# Test 4.1: Complete catalog browsing workflow
# This demonstrates the full user journey when exploring data for mapping creation

print("\n" + "="*60)
print("CATALOG BROWSING WORKFLOW")
print("="*60)

# Step 1: List all catalogs
catalogs = client.schema.list_catalogs()
print(f"\nStep 1: Found {len(catalogs)} catalogs")

if not catalogs:
    print("  No catalogs in cache (Starburst not configured)")
    print("  This is expected in environments without Starburst connection")
else:
    # Step 2: Browse first catalog
    first_catalog = catalogs[0]
    print(f"\nStep 2: Exploring catalog '{first_catalog.catalog_name}'")
    print(f"  - Schema count: {first_catalog.schema_count}")
    
    schemas = client.schema.list_schemas(first_catalog.catalog_name)
    print(f"  - Listed {len(schemas)} schemas")
    
    if schemas:
        # Step 3: Browse first schema
        first_schema = schemas[0]
        print(f"\nStep 3: Exploring schema '{first_schema.schema_name}'")
        print(f"  - Table count: {first_schema.table_count}")
        
        tables = client.schema.list_tables(
            first_catalog.catalog_name,
            first_schema.schema_name
        )
        print(f"  - Listed {len(tables)} tables")
        
        if tables:
            # Step 4: Browse first table
            first_table = tables[0]
            print(f"\nStep 4: Exploring table '{first_table.table_name}'")
            print(f"  - Table type: {first_table.table_type}")
            print(f"  - Column count: {first_table.column_count}")
            
            columns = client.schema.list_columns(
                first_catalog.catalog_name,
                first_schema.schema_name,
                first_table.table_name
            )
            print(f"  - Listed {len(columns)} columns:")
            
            # Show first 5 columns with details
            for col in columns[:5]:
                nullable = "NULL" if col.is_nullable else "NOT NULL"
                print(f"    {col.ordinal_position}. {col.column_name} {col.data_type} {nullable}")
            
            if len(columns) > 5:
                print(f"    ... and {len(columns) - 5} more columns")
        else:
            print("  - No tables in this schema")
    else:
        print("  - No schemas in this catalog")

print("\n" + "="*60)
print("Test 4.1 PASSED: Complete browsing workflow")
print("="*60)

In [ ]:
# Test 4.2: Search-based discovery workflow

print("\n" + "="*60)
print("SEARCH-BASED DISCOVERY WORKFLOW")
print("="*60)

# Search for tables with common pattern
table_results = client.schema.search_tables("a", limit=5)
print(f"\nTable search (pattern='a', limit=5): {len(table_results)} results")
if table_results:
    print("\nFound tables:")
    for r in table_results[:3]:
        print(f"  - {r.catalog_name}.{r.schema_name}.{r.table_name}")
        print(f"    Type: {r.table_type}, Columns: {r.column_count}")
    if len(table_results) > 3:
        print(f"  ... and {len(table_results) - 3} more tables")

# Search for columns with common pattern
column_results = client.schema.search_columns("a", limit=5)
print(f"\nColumn search (pattern='a', limit=5): {len(column_results)} results")
if column_results:
    print("\nFound columns:")
    for r in column_results[:3]:
        print(f"  - {r.catalog_name}.{r.schema_name}.{r.table_name}.{r.column_name}")
        print(f"    Type: {r.data_type}")
    if len(column_results) > 3:
        print(f"  ... and {len(column_results) - 3} more columns")

print("\n" + "="*60)
print("Test 4.2 PASSED: Search-based discovery workflow")
print("="*60)

## 5. Performance Validation

Verify cache operations are fast.

In [ ]:
# Test 5.1: Verify cache operations are fast

print("Cache performance validation:")

# Test catalog listing performance
start = time.time()
catalogs = client.schema.list_catalogs()
duration_ms = (time.time() - start) * 1000

print(f"\n  List catalogs: {duration_ms:.2f}ms")
assert duration_ms < 1000, f"Too slow: {duration_ms}ms (expected < 1000ms)"

# Test search performance
start = time.time()
results = client.schema.search_tables("test", limit=10)
duration_ms = (time.time() - start) * 1000

print(f"  Search tables: {duration_ms:.2f}ms")
assert duration_ms < 1000, f"Too slow: {duration_ms}ms (expected < 1000ms)"

print("\nAll cache operations completed within performance threshold")
print("\nTest 5.1 PASSED: Cache performance")

## Summary

All schema metadata tests completed successfully!

In [ ]:
# No resources to cleanup - this test is read-only
client.close()

print("\n" + "="*60)
print("SCHEMA METADATA E2E TESTS COMPLETED!")
print("="*60)

print("\nAll tests passed:")
print("\n  Section 1: Basic Listing Operations")
print("    1.1 List catalogs")
print("    1.2 Error handling for missing catalog (NotFoundError)")
print("    1.3 Error handling for missing schema (NotFoundError)")
print("    1.4 Error handling for missing table (NotFoundError)")

print("\n  Section 2: Search Operations")
print("    2.1 Search tables with pattern and limit")
print("    2.2 Search columns with pattern and limit")

print("\n  Section 3: Admin Operations")
print("    3.1 Permission checks (ForbiddenError for regular users)")
print("    3.2 Get cache statistics (admin)")
print("    3.3 Trigger cache refresh (admin)")

print("\n  Section 4: Complete Workflows")
print("    4.1 Complete catalog browsing workflow")
print("    4.2 Search-based discovery workflow")

print("\n  Section 5: Performance Validation")
print("    5.1 Cache performance (< 1000ms)")

print("\nSchema metadata tests completed.")